In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ast

df = pd.read_csv('data/tmdb_5000_movies.csv')
print(df.shape)
df.head(3)

(4803, 20)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",12/10/2009,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",5/19/2007,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bondâ€™s past sends him...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",10/26/2015,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466


In [2]:
def extract_names(text):
    try:
        items = ast.literal_eval(text)
        return ", ".join([i['name'] for i in items])
    except:
        return ""

df['genres_clean'] = df['genres'].apply(extract_names)
df['keywords_clean'] = df['keywords'].apply(extract_names)
df['year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year.astype('Int64')

df[['title', 'genres_clean', 'keywords_clean']].head(3)

,title,genres_clean,keywords_clean
0,Avatar,"Action, Adventure, Fantasy, Science Fiction","culture clash, future, space war, space colony..."
1,Pirates of the Caribbean: At World's End,"Adventure, Fantasy, Action","ocean, drug abuse, exotic island, east india t..."
2,Spectre,"Action, Adventure, Crime","spy, based on novel, secret agent, sequel, mi6..."


In [3]:
df['overview'] = df['overview'].fillna('')
df['content'] = df['genres_clean'] + ' ' + df['keywords_clean'] + ' ' + df['overview']

df[['title', 'content']].head(3)

,title,content
0,Avatar,"Action, Adventure, Fantasy, Science Fiction cu..."
1,Pirates of the Caribbean: At World's End,"Adventure, Fantasy, Action ocean, drug abuse, ..."
2,Spectre,"Action, Adventure, Crime spy, based on novel, ..."


In [4]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['content'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Map each title to its row so recommend() can find its similarity scores.
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

print("TF-IDF Matrix size:", tfidf_matrix.shape)
print("Cosine similarity Matrix size:", cosine_sim.shape)

TF-IDF Matrix size: (4803, 23291)
Cosine similarity Matrix size: (4803, 4803)


In [5]:
def recommend(title, top_n=10):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    scores = [round(s[1], 4) for s in sim_scores]

    result = df[['title', 'year', 'genres_clean', 'keywords_clean', 'vote_average']].iloc[movie_indices].copy()
    result.insert(0, 'rank', range(1, len(result) + 1))
    result['similarity_score'] = scores

    result['keywords_clean'] = result['keywords_clean'].apply(
        lambda x: ', '.join(x.split(', ')[:5]) if x else ''
    )

    result = result.rename(columns={
        'title': 'movie_title',
        'genres_clean': 'genres',
        'keywords_clean': 'keywords',
        'vote_average': 'rating'
    })
    return result.reset_index(drop=True)

recommend('Avatar')

,rank,movie_title,year,genres,keywords,rating,similarity_score
0,1,Mission to Mars,2000,Science Fiction,"mars, spacecraft, space travel, alien, long take",5.7,0.3041
1,2,Aliens,1986,"Horror, Action, Thriller, Science Fiction","android, extraterrestrial technology, space ma...",7.7,0.2869
2,3,Moonraker,1979,"Action, Adventure, Thriller, Science Fiction","venice, mass murder, space marine, space suit,...",5.9,0.2805
3,4,AlienÂ³,1992,"Science Fiction, Action, Horror","prison, android, spacecraft, space marine, imp...",6.2,0.2754
4,5,Spaceballs,1987,"Comedy, Science Fiction","android, lasergun, swordplay, temple, space ma...",6.7,0.2548
5,6,Lifeforce,1985,"Fantasy, Horror, Science Fiction, Thriller","space marine, vampire, flying saucer, comet, a...",6.2,0.2544
6,7,Treasure Planet,2002,"Adventure, Animation, Family, Fantasy, Science...","cyborg, based on novel, space marine, mutiny, ...",7.2,0.2519
7,8,Lockout,2012,"Action, Thriller, Science Fiction","usa president, anti hero, dementia, future, space",5.8,0.2505
8,9,Alien,1979,"Horror, Action, Thriller, Science Fiction","android, countdown, space marine, space suit, ...",7.9,0.2451
9,10,Planet of the Apes,2001,"Thriller, Science Fiction, Action, Adventure","gorilla, space marine, space suit, revolution,...",5.6,0.2446


In [6]:
def genre_precision_at_k(title, k=10):
    """Return the share of the top-k recommendations that share a genre."""
    if title not in indices:
        return None

    selected_genres = set(df.loc[indices[title], 'genres_clean'].split(', ')) - {''}
    if not selected_genres:
        return None

    recommended = recommend(title, k)
    matches = recommended['genres'].apply(
        lambda genres: bool(selected_genres & (set(genres.split(', ')) - {''}))
    )
    return matches.mean()

import random
random.seed(42)

valid_titles = df[df['genres_clean'].str.strip() != '']['title'].drop_duplicates().tolist()
sample_titles = random.sample(valid_titles, 100) 

precisions = []
for t in sample_titles:
    p = genre_precision_at_k(t)
    if p is not None:
        precisions.append(p)

avg_precision = sum(precisions) / len(precisions)
print(f"Averange Genre Precision@10 (n={len(precisions)}): {avg_precision:.4f}")

Averange Genre Precision@10 (n=100): 0.7830


In [7]:
def recommend(title, top_n=10):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    return df[['title', 'year', 'genres_clean', 'vote_average']].iloc[movie_indices]

recommend('Avatar')

,title,year,genres_clean,vote_average
373,Mission to Mars,2000,Science Fiction,5.7
2403,Aliens,1986,"Horror, Action, Thriller, Science Fiction",7.7
1531,Moonraker,1979,"Action, Adventure, Thriller, Science Fiction",5.9
838,AlienÂ³,1992,"Science Fiction, Action, Horror",6.2
2015,Spaceballs,1987,"Comedy, Science Fiction",6.7
1914,Lifeforce,1985,"Fantasy, Horror, Science Fiction, Thriller",6.2
305,Treasure Planet,2002,"Adventure, Animation, Family, Fantasy, Science...",7.2
2198,Lockout,2012,"Action, Thriller, Science Fiction",5.8
3158,Alien,1979,"Horror, Action, Thriller, Science Fiction",7.9
278,Planet of the Apes,2001,"Thriller, Science Fiction, Action, Adventure",5.6
